In [3]:
import duckdb
import pandas as pd
from pathlib import Path
import numpy as np

class DataHubReader:
    def __init__(self, db_name="silver.db"):
        """
        Stellt Verbindung zur DuckDB her.
        Erwartet dieselbe Ordnerstruktur wie DataHub.
        """
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data"  / db_name

        if not db_path.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {db_path}")
        self.con = duckdb.connect(str(db_path))

    def _fetch_table(self, table_name: str) -> pd.DataFrame:
        """Generische Methode zum Laden einer Tabelle."""
        return self.con.execute(f"SELECT * FROM {table_name}").df()
    def sql_command(self, command):
        return self.con.execute(f'{command}').df()
    def get_prices(self):
        return self.con.execute(f'Select * from observations').df()
    def close(self):
        """Schließt die Datenbankverbindung."""
        self.con.close()

In [4]:
hub = DataHubReader()
hub.sql_command("Show Tables;")

,name
0,entities
1,items
2,observations
3,relation_observations
4,relations


In [5]:
pricehistory = hub.sql_command("SELECT * FROM observations WHERE ItemName IN ('Open', 'Close', 'High', 'Low');")

In [6]:
pricehistory

,ObservationID,EntityID,ItemID,Date,Value,Unit,EntityCode,ItemName
0,14797,437,229,2022-02-01,19.708028,USD,ALPAT.PA,Open
1,14799,437,57,2022-03-01,19.067061,USD,ALPAT.PA,Close
2,14800,437,137,2022-03-01,19.347459,USD,ALPAT.PA,High
3,14801,437,181,2022-03-01,16.879957,USD,ALPAT.PA,Low
4,14804,437,229,2022-03-01,18.169788,USD,ALPAT.PA,Open
...,...,...,...,...,...,...,...,...
1261371,7042849,5168,229,2025-03-01,29.530001,USD,THR,Open
1261372,7043059,5168,57,2025-04-01,26.230000,USD,THR,Close
1261373,7043060,5168,137,2025-04-01,28.959999,USD,THR,High
1261374,7043061,5168,181,2025-04-01,23.049999,USD,THR,Low


In [7]:
def price_to_return(df_close: pd.DataFrame, log_returns: bool = False) -> pd.DataFrame:

    #df_close["Date"] = pd.to_datetime(df_close["Date"])
    df_close = df_close.sort_values(["EntityCode", "Date"])
    
    # 3. Returns berechnen (pro Firma)
    if log_returns:
        df_close["Return"] = (
            df_close.groupby("EntityCode")["Value"]
            .transform(lambda x: (x / x.shift(1)).apply(lambda r: pd.NA if pd.isna(r) else pd.np.log(r)))
        )
    else:
        df_close["Return"] = (
            df_close.groupby("EntityCode")["Value"]
            .pct_change()
        )
    
    return df_close

In [8]:
df = price_to_return(pricehistory[pricehistory['ItemName'] == 'Close'])
df = df[df['Date'].dt.day == 1]
df

,ObservationID,EntityID,ItemID,Date,Value,Unit,EntityCode,ItemName,Return
108127,5555,1,57,2021-05-01,11.867758,USD,2020.OL,Close,NaN
108131,5561,1,57,2021-06-01,12.707325,USD,2020.OL,Close,0.070743
108135,5567,1,57,2021-07-01,12.189383,USD,2020.OL,Close,-0.040759
108139,5573,1,57,2021-08-01,14.433397,USD,2020.OL,Close,0.184096
108143,5579,1,57,2021-09-01,14.375350,USD,2020.OL,Close,-0.004022
...,...,...,...,...,...,...,...,...,...
956517,7008582,5776,57,2025-12-01,26.330000,USD,ZYME,Close,-0.014227
956521,7008756,5776,57,2026-01-01,22.530001,USD,ZYME,Close,-0.144322
956525,7008762,5776,57,2026-02-01,23.290001,USD,ZYME,Close,0.033733
956529,7008767,5776,57,2026-03-01,25.040001,USD,ZYME,Close,0.075140


In [9]:
pd.set_option('display.max_rows', 10)

In [10]:
df = df[['Date', 'EntityCode', 'Return']]
df['Date'] = pd.to_datetime(df['Date'])
# Pivot: Zeit × Assets
returns_matrix = df.pivot(index='Date', columns='EntityCode', values='Return')
# Sortieren
returns_matrix = returns_matrix.sort_index()

returns_matrix_clean = returns_matrix.copy()

# harte Bereinigung
returns_matrix_clean = returns_matrix_clean.replace([np.inf, -np.inf], np.nan)
returns_matrix_clean = returns_matrix_clean.fillna(0)

# Normalisierung pro Asset/Spalte
mean = returns_matrix_clean.mean()
std = returns_matrix_clean.std().replace(0, 1)

returns_matrix_norm = (returns_matrix_clean - mean) / std
returns_matrix_norm = returns_matrix_norm.replace([np.inf, -np.inf], np.nan).fillna(0)
returns_matrix = returns_matrix_norm

C:\Users\Konra\AppData\Local\Temp\ipykernel_5760\4093514450.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date'])


In [11]:
corr_matrix = returns_matrix.corr()
corr_matrix

EntityCode,2020.OL,5PG.OL,74SW.PA,8GW.IR,A,A2A.MI,A5G.IR,AA,AAA.PA,AAL,...,ZSTK,ZTEK,ZTS,ZUC.MI,ZUMZ,ZV.MI,ZVIA,ZVRA,ZWS,ZYME
EntityCode,,,,,,,,,,,,,,,,,,,,,
2020.OL,1.000000,0.103076,0.163314,-0.275424,0.158038,0.248050,0.322395,0.161501,0.174267,0.109413,...,0.055529,-0.003990,0.092080,0.244056,-0.056455,0.224840,-0.287283,-0.092133,0.138487,0.070352
5PG.OL,0.103076,1.000000,-0.241898,-0.033404,-0.014650,-0.084946,0.137660,0.076670,-0.015705,-0.150292,...,-0.151519,-0.195063,-0.237666,-0.100088,-0.158473,-0.053705,-0.094877,-0.124175,-0.304787,-0.009582
74SW.PA,0.163314,-0.241898,1.000000,0.074609,0.012513,0.207361,0.107696,0.179974,-0.053002,0.281178,...,0.122622,0.056760,0.291552,-0.001853,0.129075,0.353001,-0.024357,0.094132,0.156181,0.244964
8GW.IR,-0.275424,-0.033404,0.074609,1.000000,-0.132982,0.009510,-0.204974,-0.089283,-0.029706,-0.289922,...,-0.058348,0.143438,-0.021449,-0.102352,0.053871,-0.174946,-0.053884,-0.069718,-0.134733,-0.108726
A,0.158038,-0.014650,0.012513,-0.132982,1.000000,0.371390,0.151050,0.336016,-0.047521,0.358442,...,0.217572,0.004104,0.490386,0.245432,0.357788,0.477623,0.210034,0.199673,0.402079,0.488523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZV.MI,0.224840,-0.053705,0.353001,-0.174946,0.477623,0.442659,0.263244,0.179397,0.011010,0.320603,...,0.165049,-0.068530,0.508094,0.270676,0.104294,1.000000,-0.074931,0.205629,0.226291,0.288498
ZVIA,-0.287283,-0.094877,-0.024357,-0.053884,0.210034,-0.004311,0.000857,-0.100060,-0.001240,0.328284,...,0.026802,0.047896,-0.055767,-0.085988,0.058112,-0.074931,1.000000,0.219008,0.096623,0.281641
ZVRA,-0.092133,-0.124175,0.094132,-0.069718,0.199673,0.046118,0.015884,-0.033274,-0.006925,0.162393,...,0.056873,0.094121,0.326286,0.086259,0.289822,0.205629,0.219008,1.000000,0.199765,0.314363


In [12]:
corr = corr_matrix.values
corr

array([[ 1.        ,  0.10307555,  0.16331369, ..., -0.09213324,
         0.13848703,  0.0703523 ],
       [ 0.10307555,  1.        , -0.24189835, ..., -0.12417453,
        -0.30478683, -0.00958183],
       [ 0.16331369, -0.24189835,  1.        , ...,  0.0941315 ,
         0.1561807 ,  0.24496358],
       ...,
       [-0.09213324, -0.12417453,  0.0941315 , ...,  1.        ,
         0.19976486,  0.31436322],
       [ 0.13848703, -0.30478683,  0.1561807 , ...,  0.19976486,
         1.        ,  0.12434295],
       [ 0.0703523 , -0.00958183,  0.24496358, ...,  0.31436322,
         0.12434295,  1.        ]])

In [13]:
def build_edges_threshold(corr, threshold=0.8):
    num_nodes = corr.shape[0]
    edge_index = []

    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):  # 🔥 wichtig!
            if abs(corr[i, j]) > threshold:
                edge_index.append([i, j])
                edge_index.append([j, i])

    return np.array(edge_index).T

edges = build_edges_threshold(corr)

In [14]:
print(edges.shape) 

(2, 10396)


In [15]:
import torch
from torch_geometric.data import Data

edge_index = torch.tensor(edges, dtype=torch.long)
print(edge_index.shape) 

c:\Users\Konra\miniforge3\envs\gnn-data\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.Size([2, 10396])


In [21]:
def create_pyg_graphs(returns_matrix, edge_index, lookback=5, horizon=1):
    graphs = []

    values = returns_matrix.fillna(0).values
    dates = returns_matrix.index
    num_timesteps, num_assets = values.shape

    for t in range(lookback, num_timesteps - horizon):
        # vergangene Returns
        x_window = values[t - lookback:t]      # [lookback, num_assets]
        x = torch.tensor(x_window.T, dtype=torch.float)  # [num_assets, lookback]

        # Ziel: zukünftiger Return
        y_future = values[t + horizon]         # [num_assets]
        y = torch.tensor(y_future, dtype=torch.float)

        graph = Data(
            x=x,
            edge_index=edge_index,
            y=y,
            date=dates[t]
        )

        graphs.append(graph)

    return graphs


In [22]:
graphs = create_pyg_graphs(
    returns_matrix=returns_matrix,
    edge_index=edge_index,
    lookback=5,
    horizon=1
)

print(len(graphs))
print(graphs[0])
print(graphs[0].x.shape)
print(graphs[0].y.shape)

54
Data(x=[5738, 5], edge_index=[2, 10396], y=[5738], date=2021-10-01 00:00:00)
torch.Size([5738, 5])
torch.Size([5738])


In [23]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(graphs, batch_size=16, shuffle=False)

In [24]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class ReturnGCN(torch.nn.Module):
    def __init__(self, lookback, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(lookback, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.out = torch.nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.out(x).squeeze(-1)
        return x

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ReturnGCN(lookback=5, hidden_dim=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1000):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        pred = model(batch)
        loss = F.mse_loss(pred, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {avg_loss:.6f}")

Epoch 0 | Loss: 1.107536
Epoch 100 | Loss: 1.066707
Epoch 200 | Loss: 1.064458
Epoch 300 | Loss: 1.062859
Epoch 400 | Loss: 1.061658
Epoch 500 | Loss: 1.060592
Epoch 600 | Loss: 1.059630
Epoch 700 | Loss: 1.058751
Epoch 800 | Loss: 1.057969
Epoch 900 | Loss: 1.057274
